# Targeted Dropout Ladder: Stage B and Gated Stage C

This notebook continues `tasks/TASK_targeted_dropout_uncertainty_ladder.md` after the
corrected Stage A result. It runs the fixed three-config 100K confirmation first, writes
an auditable promotion manifest, and refuses to start the official 500K stage unless that
manifest passes the hard gates. Raw `K` samples, source-row identity, runtime configs,
shard logs, and compact strategy outputs are retained.

Start with `RUN_STAGE = "stage_b_100k"`. Stage C is a separate, explicit run after Stage B
review; it is never launched automatically by the Stage B execution path.

## 0. Resource Assumptions

- Runtime: Google Colab with Python 3.12 and PyTorch 2.10.x or 2.11.x. The current runtime is preferred; past runtime 2026.04 is also supported.
- GPU: T4, L4, A100, and comparable CUDA accelerators are supported.
- Source checkpoints and official 500K artifacts already exist under the project Google Drive root.
- Local scratch: about 1.2 GB for the Stage C token copy, plus Python/package overhead.
- Drive: allow roughly 1.5 GB for Stage B or 4 GB for a two-config Stage C run. The scorer in
  the pinned commit must use bounded shard index files; the preflight cell enforces this.
- Default production shard size is 24,992 rows, matching the prior successful 500K run.
  Each shard is independently validated and restartable after a disconnect.
- Expected wall time from the earlier T4 run is roughly 2-3 hours for all three Stage B
  configs and 3-7 hours for one or two Stage C configs. Faster GPUs reduce this materially.
- Keep the Colab tab open while a cell is launching jobs. Completed shard outputs remain on Drive.

## 1. Runtime and Drive

In [ ]:
# PYTHON CELL
from google.colab import drive
drive.mount('/content/drive')

import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if not gpu:
    raise RuntimeError('No CUDA GPU is visible. Select a GPU runtime before continuing.')
print('python:', sys.version)
print('platform:', platform.platform())
print('gpu:', gpu)
print('local disk free GB:', round(shutil.disk_usage('/content').free / 1e9, 2))

In [ ]:
# PYTHON CELL
DRIVE = Path('/content/drive/MyDrive/color-filter-ablation')
DATA_DRIVE = DRIVE / 'data'
SOURCE_RESULTS_DRIVE = DRIVE / 'results'
MODELS_DRIVE = DRIVE / 'assets' / 'raw' / 'models'
if not DRIVE.exists():
    raise FileNotFoundError(f'Project Drive root not found: {DRIVE}')

print('project Drive root:', DRIVE)
print('Drive disk free GB:', round(shutil.disk_usage('/content/drive').free / 1e9, 2))

## 2. Clone, Pin, and Install

In [ ]:
# PYTHON CELL
import re

OLMO_REPO = 'https://github.com/myazdani/color-filter-olmo.git'
OLMO_SHA = '98e980a82036f451eed1a0bc3c790c3fef606318'
OLMO_DIR = Path('/content/color-filter-olmo')

if not re.fullmatch(r'[0-9a-f]{40}', OLMO_SHA):
    raise RuntimeError(
        'Set OLMO_SHA to the full pushed commit containing the score-index alignment fix, '
        'and bounded DictMemmapWriter support. Do not run from an unpinned branch.'
    )

if not (OLMO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', OLMO_REPO, str(OLMO_DIR)], check=True)
subprocess.run(['git', '-C', str(OLMO_DIR), 'fetch', 'origin', OLMO_SHA], check=True)
subprocess.run(['git', '-C', str(OLMO_DIR), 'checkout', '--detach', OLMO_SHA], check=True)
checked_out = subprocess.run(
    ['git', '-C', str(OLMO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
if checked_out != OLMO_SHA:
    raise RuntimeError(f'Checkout mismatch: expected {OLMO_SHA}, found {checked_out}')
print('pinned OLMo commit:', checked_out)

In [ ]:
# PYTHON CELL
# Small, explicit overlay known to work with the repository's Colab scoring path.
from packaging.requirements import Requirement
from packaging.version import Version
import torch as colab_torch

COLAB_PYTHON = (3, 12)
COLAB_TORCH_MIN = Version('2.10.0')
COLAB_TORCH_MAX = Version('2.12.0')

def validate_colab_runtime(torch_module, phase):
    torch_version = Version(torch_module.__version__.split('+', 1)[0])
    if sys.version_info[:2] != COLAB_PYTHON or not (COLAB_TORCH_MIN <= torch_version < COLAB_TORCH_MAX):
        raise RuntimeError(
            'This notebook requires Python 3.12 and PyTorch 2.10.x or 2.11.x. '
            f'Found Python {sys.version_info.major}.{sys.version_info.minor}, PyTorch {torch_module.__version__}. '
            'Select a supported Colab runtime before installing packages.'
        )
    print(f'accepted Colab runtime ({phase}):', f'Python {sys.version_info.major}.{sys.version_info.minor}', f'PyTorch {torch_module.__version__}')
    return torch_version

colab_torch_version = validate_colab_runtime(colab_torch, 'before installation')

pyproject_path = OLMO_DIR / 'pyproject.toml'
pyproject_text = pyproject_path.read_text()
legacy_torch_requirement = '    "torch>=2.1,<2.3",'
colab_torch_requirements = (
    "    \"torch>=2.1,<2.3; python_version < '3.12'\",\n"
    "    \"torch>=2.10,<2.12; python_version >= '3.12'\","
)
if legacy_torch_requirement in pyproject_text:
    pyproject_path.write_text(pyproject_text.replace(legacy_torch_requirement, colab_torch_requirements))
elif not all(requirement in pyproject_text for requirement in colab_torch_requirements.splitlines()):
    raise RuntimeError('Pinned pyproject has an unexpected Torch requirement; update the explicit Colab overlay.')
print('applied verified Colab Torch metadata overlay:', colab_torch_requirements.replace('\n', ' | '))

PIP_OVERLAY = [
    'omegaconf==2.3.0',
    'rich==13.9.4',
    'cached_path==1.8.10',
    'packaging==24.2',
    'boto3==1.35.94',
    'google-cloud-storage==2.19.0',
    'wandb==0.19.1',
    'torchmetrics==1.6.1',
    'datasets==3.2.0',
    'huggingface_hub==0.27.1',
    'transformers==4.47.1',
    'tokenizers==0.21.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PIP_OVERLAY], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(OLMO_DIR)],
    check=True,
)

from importlib.metadata import requires
installed_torch_requirements = [item for item in (requires('ai2-olmo') or []) if Requirement(item).name.lower() == 'torch']
parsed_torch_requirements = [Requirement(item) for item in installed_torch_requirements]
active_torch_requirements = [req for req in parsed_torch_requirements if req.marker is None or req.marker.evaluate()]
expected_torch_specifier = Requirement('torch>=2.10,<2.12').specifier
if not any(req.specifier == expected_torch_specifier for req in active_torch_requirements):
    raise RuntimeError(f'Installed OLMo metadata does not declare Colab Torch support: {installed_torch_requirements}')
print('installed OLMo Torch requirements:', installed_torch_requirements)

sys.path.insert(0, str(OLMO_DIR))
import numpy as np
import pandas as pd
import pyarrow
import torch
from omegaconf import OmegaConf
from olmo.config import TrainConfig

torch_version = validate_colab_runtime(torch, 'after installation')
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'available:', torch.cuda.is_available())
print('numpy:', np.__version__, 'pandas:', pd.__version__, 'pyarrow:', pyarrow.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access CUDA after installation.')

RUNTIME_IDENTITY = {
    'python': f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}',
    'torch': torch.__version__, 'cuda': torch.version.cuda,
}
print('runtime identity:', RUNTIME_IDENTITY)

In [ ]:
# PYTHON CELL
# Producer capability probes. These fail before any expensive scoring if an old commit was pinned.
metrics_source = (OLMO_DIR / 'scripts/21_dropout_uncertainty_metrics.py').read_text()
score_source = (OLMO_DIR / 'olmo/score.py').read_text()
writer_source = (OLMO_DIR / 'olmo/data/dict_memmap_dataset.py').read_text()
required_markers = {
    'score-index table alignment': 'metadata_and_full_scores_indexed_by_score_index' in metrics_source,
    'bounded scorer index allocation': 'max_entries=' in score_source,
    'shard-continuous stochastic seeds': 'data_start_step or 0' in score_source,
    'bounded writer capacity': 'max_entries: Optional[int]' in writer_source,
}
missing = [name for name, present in required_markers.items() if not present]
if missing:
    raise RuntimeError('Pinned producer commit lacks required capabilities: ' + ', '.join(missing))

for script in ['scripts/21_dropout_uncertainty_metrics.py', 'scripts/22_dropout_strategy_sweep.py']:
    probe = subprocess.run(
        [sys.executable, str(OLMO_DIR / script), '--help'],
        cwd=str(OLMO_DIR),
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(f'CLI probe failed for {script}:\n{probe.stderr[-2000:]}')
print('capability probes:', required_markers)

## 3. Configure Paths and Stage Gate

In [ ]:
# PYTHON CELL
import json
from datetime import datetime, timezone

RUN_STAGE = 'stage_b_100k'  # allowed: stage_b_100k, stage_c_500k
ENABLE_STAGE_C = False
STAGE_C_CONFIG_IDS = []  # optional 1-2 ID override; must have passed Stage B

NUM_SAMPLES = 8
SEED = 1
SUBSET_SELECTION_SEED = 1729
SEQ_LEN = 512
GLOBAL_BATCH_SIZE = 32
MICROBATCH = 16
SHARD_ROWS = 24_992
SMOKE_ROWS = 320
BENCH_ROWS = 640
TAU64_CUTOFF = 0.3513622284
FILE_SEQS = 1_048_576

TOKENS_DRIVE = DATA_DRIVE / 'score_pool_tokens_official_500k.npy'
META_DRIVE = DATA_DRIVE / 'score_pool_meta_official_500k.parquet'
FULL_SCORES_DRIVE = SOURCE_RESULTS_DRIVE / 'score-pool-robustness-official-500k' / 'scores_full.parquet'
PAIR_MID2_DRIVE = SOURCE_RESULTS_DRIVE / 'score-pool-robustness-official-500k' / 'scores_pair_mid2.parquet'
BROAD_DROPOUT_ANALYSIS = SOURCE_RESULTS_DRIVE / 'dropout-uncertainty' / 'dropout_k8_p005' / 'analysis'
PRIOR_CHECKPOINT = MODELS_DRIVE / 'prior'
BOOKS_CHECKPOINT = MODELS_DRIVE / 'conditional_books'

CONFIG_BY_ID = {
    'dropout_attn_p001': {
        'config_id': 'dropout_attn_p001', 'dropout_target': 'attention',
        'attention_dropout': 0.01, 'residual_dropout': 0.0, 'embedding_dropout': 0.0,
        'purpose': 'Stage A best attention-only target',
    },
    'dropout_resid_p001': {
        'config_id': 'dropout_resid_p001', 'dropout_target': 'residual',
        'attention_dropout': 0.0, 'residual_dropout': 0.01, 'embedding_dropout': 0.0,
        'purpose': 'Stage A best residual-or-combined target',
    },
    'dropout_embed_p0005': {
        'config_id': 'dropout_embed_p0005', 'dropout_target': 'embedding',
        'attention_dropout': 0.0, 'residual_dropout': 0.0, 'embedding_dropout': 0.005,
        'purpose': 'Stage A best overall rank-preserving target',
    },
}
STAGE_B_IDS = ['dropout_attn_p001', 'dropout_resid_p001', 'dropout_embed_p0005']
STAGE_B_ROOT = SOURCE_RESULTS_DRIVE / 'dropout-uncertainty-targeted' / 'stage_b_100k'
STAGE_B_ACCEPTANCE = STAGE_B_ROOT / 'stage_acceptance.json'

if RUN_STAGE == 'stage_b_100k':
    SUBSET_ID = RUN_STAGE
    STAGE_ROWS = 100_000
    TARGET_CONFIGS = [CONFIG_BY_ID[cid] for cid in STAGE_B_IDS]
elif RUN_STAGE == 'stage_c_500k':
    if not ENABLE_STAGE_C:
        raise RuntimeError('Stage C is locked. Set ENABLE_STAGE_C=True only after reviewing Stage B acceptance.')
    if not STAGE_B_ACCEPTANCE.exists():
        raise FileNotFoundError(f'Stage B acceptance manifest is required: {STAGE_B_ACCEPTANCE}')
    stage_b_gate = json.loads(STAGE_B_ACCEPTANCE.read_text())
    if not stage_b_gate.get('stage_passed'):
        raise RuntimeError('Stage B did not pass; Stage C must not run.')
    if stage_b_gate.get('olmo_sha') != OLMO_SHA:
        raise RuntimeError(
            'Stage B and Stage C must use the same pinned producer commit. '
            f"Stage B used {stage_b_gate.get('olmo_sha')}, current pin is {OLMO_SHA}."
        )
    allowed = stage_b_gate.get('promoted_config_ids', [])
    chosen = STAGE_C_CONFIG_IDS or stage_b_gate.get('recommended_stage_c_config_ids', [])
    if not 1 <= len(chosen) <= 2:
        raise RuntimeError(f'Stage C requires 1-2 confirmed configs, found: {chosen}')
    if len(set(chosen)) != len(chosen) or any(cid not in allowed for cid in chosen):
        raise RuntimeError(f'Stage C configs must be distinct Stage B promotions. allowed={allowed}, chosen={chosen}')
    SUBSET_ID = RUN_STAGE
    STAGE_ROWS = 500_000
    TARGET_CONFIGS = [CONFIG_BY_ID[cid] for cid in chosen]
else:
    raise ValueError(f'Unsupported RUN_STAGE: {RUN_STAGE}')

STAGE_ROOT = SOURCE_RESULTS_DRIVE / 'dropout-uncertainty-targeted' / SUBSET_ID
RAW_SCORE_DRIVE = STAGE_ROOT / 'raw_score_shards'
CONFIG_DRIVE = DRIVE / 'runtime_configs' / 'dropout-uncertainty-targeted' / SUBSET_ID
SUBSET_MANIFEST = STAGE_ROOT / 'subset_manifest.json'
SOURCE_ROWS_DRIVE = STAGE_ROOT / 'subset_source_rows.npy'
RUN_STATE_PATH = STAGE_ROOT / 'run_state.json'
REPORT_DRIVE = STAGE_ROOT / 'report'
LOCAL_WORK = Path('/content/targeted_dropout_followup') / SUBSET_ID
RUNTIME_CONFIG_DIR = LOCAL_WORK / 'runtime_configs'
RUNTIME_CHECKPOINT_DIR = LOCAL_WORK / 'runtime_checkpoints'
for path in [STAGE_ROOT, RAW_SCORE_DRIVE, CONFIG_DRIVE, REPORT_DRIVE, LOCAL_WORK, RUNTIME_CONFIG_DIR, RUNTIME_CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('stage:', RUN_STAGE, 'rows:', STAGE_ROWS, 'K:', NUM_SAMPLES)
print('configs:', [item['config_id'] for item in TARGET_CONFIGS])
print('stage root:', STAGE_ROOT)

## 4. Validate Inputs and Build the Fixed Subset

In [ ]:
# PYTHON CELL
import hashlib

def canonical_sha256(payload):
    encoded = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()

def sampled_file_identity(path, sample_bytes=1_048_576):
    path = Path(path)
    stat = path.stat()
    digest = hashlib.sha256()
    digest.update(str(stat.st_size).encode('ascii'))
    with path.open('rb') as handle:
        digest.update(handle.read(sample_bytes))
        if stat.st_size > sample_bytes:
            handle.seek(max(0, stat.st_size - sample_bytes))
            digest.update(handle.read(sample_bytes))
    return {'path': str(path), 'bytes': stat.st_size, 'sampled_sha256': digest.hexdigest()}

required_inputs = [
    TOKENS_DRIVE, META_DRIVE, FULL_SCORES_DRIVE,
    PRIOR_CHECKPOINT / 'model.pt', PRIOR_CHECKPOINT / 'config.yaml',
    BOOKS_CHECKPOINT / 'model.pt', BOOKS_CHECKPOINT / 'config.yaml',
]
if RUN_STAGE == 'stage_c_500k':
    required_inputs.extend([
        PAIR_MID2_DRIVE,
        BROAD_DROPOUT_ANALYSIS / 'mc_samples_dropout_k8_p005.npz',
        BROAD_DROPOUT_ANALYSIS / 'strategy' / 'strategy_sweep_metrics.csv',
        BROAD_DROPOUT_ANALYSIS / 'strategy' / 'strategy_selection_overlap.csv',
    ])
missing = [str(path) for path in required_inputs if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required Drive inputs:\n' + '\n'.join(missing))

SOURCE_TOKEN_IDENTITY = sampled_file_identity(TOKENS_DRIVE)
CHECKPOINT_IDENTITIES = {}
for model_id, checkpoint in [('prior', PRIOR_CHECKPOINT), ('books', BOOKS_CHECKPOINT)]:
    model_file = checkpoint / 'model.pt'
    config_file = checkpoint / 'config.yaml'
    if model_file.stat().st_size <= 0 or config_file.stat().st_size <= 0:
        raise ValueError(f'Empty checkpoint artifact under {checkpoint}')
    CHECKPOINT_IDENTITIES[model_id] = {
        'model': sampled_file_identity(model_file),
        'config': sampled_file_identity(config_file, sample_bytes=config_file.stat().st_size),
    }
if CHECKPOINT_IDENTITIES['prior']['model']['sampled_sha256'] == CHECKPOINT_IDENTITIES['books']['model']['sampled_sha256']:
    raise RuntimeError('Prior and Books checkpoints have the same sampled fingerprint')

source_tokens = np.load(TOKENS_DRIVE, mmap_mode='r')
source_meta = pd.read_parquet(META_DRIVE)
source_full = pd.read_parquet(FULL_SCORES_DRIVE)
if source_tokens.ndim != 2 or source_tokens.shape[1] != SEQ_LEN:
    raise ValueError(f'Expected token shape (N, {SEQ_LEN}), found {source_tokens.shape}')
if source_tokens.dtype not in (np.dtype(np.int32), np.dtype(np.uint32)):
    raise ValueError(f'Expected official tokens to use int32 or uint32, found {source_tokens.dtype}')
source_rows = source_tokens.shape[0]
if source_rows != 500_000 or len(source_meta) != source_rows or len(source_full) != source_rows:
    raise ValueError(
        f'Official source row mismatch: tokens={source_rows}, metadata={len(source_meta)}, full={len(source_full)}'
    )
if 'pool_name' not in source_meta.columns:
    raise ValueError('Official metadata is missing pool_name')
full_color_columns = [
    col for col in ['full_color_score', 'color', 'color_score', 'ablated_color_score']
    if col in source_full.columns
]
if not full_color_columns:
    raise ValueError(f'Could not identify deterministic full color column: {source_full.columns.tolist()}')

expected_pools = [
    'hard_positive', 'hard_negative', 'random_positive', 'random_negative', 'tail_negative'
]
source_pool_counts = source_meta['pool_name'].value_counts().to_dict()
if any(source_pool_counts.get(pool, 0) < 20_000 for pool in expected_pools):
    raise ValueError(f'Official pool counts cannot support balanced Stage B: {source_pool_counts}')

if RUN_STAGE == 'stage_b_100k':
    rng = np.random.Generator(np.random.PCG64(SUBSET_SELECTION_SEED))
    selected_parts = []
    for pool in expected_pools:
        candidates = np.flatnonzero(source_meta['pool_name'].to_numpy() == pool)
        selected_parts.append(rng.choice(candidates, size=20_000, replace=False))
    selected_source_rows = np.sort(np.concatenate(selected_parts).astype(np.int64))
else:
    selected_source_rows = np.arange(source_rows, dtype=np.int64)

if len(selected_source_rows) != STAGE_ROWS or len(np.unique(selected_source_rows)) != STAGE_ROWS:
    raise RuntimeError('Subset source-row selection is not complete and unique')

subset_meta_frame = source_meta.iloc[selected_source_rows].copy().reset_index(drop=True)
subset_full_frame = source_full.iloc[selected_source_rows].copy().reset_index(drop=True)
subset_meta_frame['source_row'] = selected_source_rows
subset_full_frame['source_row'] = selected_source_rows
subset_pool_counts = subset_meta_frame['pool_name'].value_counts().sort_index().to_dict()
if RUN_STAGE == 'stage_b_100k' and set(subset_pool_counts.values()) != {20_000}:
    raise RuntimeError(f'Stage B subset is not balanced: {subset_pool_counts}')

source_rows_sha256 = hashlib.sha256(selected_source_rows.tobytes()).hexdigest()
subset_meta = LOCAL_WORK / f'{SUBSET_ID}_meta.parquet'
subset_full = LOCAL_WORK / f'{SUBSET_ID}_full_scores.parquet'
subset_raw = LOCAL_WORK / f'{SUBSET_ID}_tokens.uint32.raw'
subset_raw_state = LOCAL_WORK / f'{SUBSET_ID}_tokens.uint32.manifest.json'
subset_meta_frame.to_parquet(subset_meta, index=False)
subset_full_frame.to_parquet(subset_full, index=False)
np.save(SOURCE_ROWS_DRIVE, selected_source_rows)

expected_raw_bytes = STAGE_ROWS * SEQ_LEN * np.dtype(np.uint32).itemsize
subset_cache_payload = {
    'subset_id': SUBSET_ID, 'rows': STAGE_ROWS, 'seq_len': SEQ_LEN, 'dtype': 'uint32',
    'source_rows_sha256': source_rows_sha256, 'source_token_identity': SOURCE_TOKEN_IDENTITY,
}
subset_cache_fingerprint = canonical_sha256(subset_cache_payload)
try:
    cached_state = json.loads(subset_raw_state.read_text()) if subset_raw_state.exists() else {}
except (OSError, json.JSONDecodeError):
    cached_state = {}
cache_matches = (
    subset_raw.exists() and subset_raw.stat().st_size == expected_raw_bytes
    and cached_state.get('fingerprint') == subset_cache_fingerprint
)
if not cache_matches:
    temp_raw = subset_raw.with_suffix('.tmp')
    with temp_raw.open('wb') as handle:
        for start in range(0, STAGE_ROWS, 4096):
            rows = selected_source_rows[start:start + 4096]
            np.asarray(source_tokens[rows], dtype=np.uint32).tofile(handle)
    os.replace(temp_raw, subset_raw)
    subset_raw_state.write_text(json.dumps({
        'fingerprint': subset_cache_fingerprint, **subset_cache_payload,
    }, indent=2, sort_keys=True) + '\n')
if subset_raw.stat().st_size != expected_raw_bytes:
    raise RuntimeError(f'Local raw token copy has wrong size: {subset_raw.stat().st_size}')

SUBSET_FINGERPRINT = canonical_sha256({
    **subset_cache_payload, 'pool_counts': subset_pool_counts,
    'full_score_column': full_color_columns[0],
})
subset_manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'subset_id': SUBSET_ID,
    'row_count': STAGE_ROWS,
    'pool_counts': subset_pool_counts,
    'selection_method': 'all_official_rows' if RUN_STAGE == 'stage_c_500k' else 'balanced_without_replacement',
    'selection_seed': None if RUN_STAGE == 'stage_c_500k' else SUBSET_SELECTION_SEED,
    'source_rows_path': str(SOURCE_ROWS_DRIVE),
    'source_rows_sha256': source_rows_sha256,
    'source_token_identity': SOURCE_TOKEN_IDENTITY,
    'checkpoint_identities': CHECKPOINT_IDENTITIES,
    'runtime_identity': RUNTIME_IDENTITY,
    'subset_fingerprint': SUBSET_FINGERPRINT,
    'source_metadata_path': str(META_DRIVE),
    'source_token_path': str(TOKENS_DRIVE),
    'full_score_reference_path': str(FULL_SCORES_DRIVE),
    'full_score_column': full_color_columns[0],
    'same_subset_for_every_config': True,
}
SUBSET_MANIFEST.write_text(json.dumps(subset_manifest, indent=2, sort_keys=True) + '\n')
print(json.dumps(subset_manifest, indent=2, sort_keys=True))
print('local raw GB:', round(subset_raw.stat().st_size / 1e9, 3))

In [ ]:
# PYTHON CELL
if SHARD_ROWS % GLOBAL_BATCH_SIZE != 0:
    raise ValueError('SHARD_ROWS must be divisible by GLOBAL_BATCH_SIZE')
if STAGE_ROWS % GLOBAL_BATCH_SIZE != 0:
    raise ValueError('STAGE_ROWS must be divisible by GLOBAL_BATCH_SIZE')

SHARDS = []
start = 0
while start < STAGE_ROWS:
    rows = min(SHARD_ROWS, STAGE_ROWS - start)
    if rows % GLOBAL_BATCH_SIZE != 0:
        raise ValueError(f'Final shard is not batch aligned: start={start}, rows={rows}')
    SHARDS.append({
        'start': start,
        'end': start + rows,
        'rows': rows,
        'data_start_step': start // GLOBAL_BATCH_SIZE,
    })
    start += rows

shard_plan_path = STAGE_ROOT / 'shard_plan.json'
shard_plan_path.write_text(json.dumps(SHARDS, indent=2, sort_keys=True) + '\n')
print('shards:', len(SHARDS), 'first:', SHARDS[0], 'last:', SHARDS[-1])
print('bounded index bytes per full shard:', SHARD_ROWS * np.dtype(np.int64).itemsize)

## 5. Scoring Helpers

In [ ]:
# PYTHON CELL
import os
import shutil
import time

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('WANDB_MODE', 'disabled')
TEMPLATE_CONFIG = OLMO_DIR / 'configs/sweeps/score-targeted-dropout-uncertainty.yaml'

def symlink_or_refresh(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.exists():
        raise FileNotFoundError(src)
    if dst.exists() or dst.is_symlink():
        try:
            if dst.resolve() == src.resolve():
                return
        except FileNotFoundError:
            pass
        dst.unlink()
    os.symlink(src, dst)

def prepare_model_only_checkpoint(checkpoint_path):
    source = Path(checkpoint_path)
    runtime = RUNTIME_CHECKPOINT_DIR / source.name
    runtime.mkdir(parents=True, exist_ok=True)
    symlink_or_refresh(source / 'model.pt', runtime / 'model.pt')
    symlink_or_refresh(source / 'config.yaml', runtime / 'config.yaml')
    for state_name in ['train.pt', 'other.pt']:
        state_path = runtime / state_name
        if not state_path.exists():
            torch.save({}, state_path)
    return runtime

def load_checkpoint_score_config(checkpoint_path):
    checkpoint_cfg = OmegaConf.load(Path(checkpoint_path) / 'config.yaml')
    cfg = OmegaConf.load(TEMPLATE_CONFIG)
    cfg.model = checkpoint_cfg.model
    if 'tokenizer' in checkpoint_cfg:
        cfg.tokenizer = checkpoint_cfg.tokenizer
    if 'targeted_ladder' in cfg:
        del cfg['targeted_ladder']
    return cfg

def build_score_config(config, model_id, checkpoint_path, output_dir, shard, microbatch, console_log_interval=25):
    rows = int(shard['rows'])
    cfg = load_checkpoint_score_config(checkpoint_path)
    cfg.run_name = f"{config['config_id']}_{model_id}_{shard['start']:06d}_{shard['end']:06d}"
    cfg.save_folder = str(output_dir)
    cfg.load_path = str(prepare_model_only_checkpoint(checkpoint_path))
    cfg.load_checkpoint_type = 'unsharded'
    cfg.max_duration = rows // GLOBAL_BATCH_SIZE
    cfg.data_start_step = int(shard['data_start_step'])
    cfg.global_train_batch_size = GLOBAL_BATCH_SIZE
    cfg.device_train_batch_size = GLOBAL_BATCH_SIZE
    cfg.device_train_microbatch_size = microbatch
    cfg.data.paths = [str(subset_raw)]
    cfg.data.memmap_dtype = 'uint32'
    cfg.data.num_workers = 0
    cfg.seed = SEED
    cfg.model.attention_dropout = float(config['attention_dropout'])
    cfg.model.residual_dropout = float(config['residual_dropout'])
    cfg.model.embedding_dropout = float(config['embedding_dropout'])
    cfg.uncertainty_scoring.enabled = True
    cfg.uncertainty_scoring.num_samples = NUM_SAMPLES
    cfg.uncertainty_scoring.perturbation_type = 'dropout'
    cfg.uncertainty_scoring.coupled_masks = True
    cfg.restore_dataloader = False
    cfg.reset_optimizer_state = True
    cfg.reset_trainer_state = True
    cfg.console_log_interval = min(max(1, int(console_log_interval)), max(1, int(cfg.max_duration)))
    cfg.gen1_gc_interval = 50
    cfg.save_data_indices = True
    return cfg

def write_config(cfg, name):
    local_path = RUNTIME_CONFIG_DIR / name
    local_path.parent.mkdir(parents=True, exist_ok=True)
    OmegaConf.save(cfg, local_path)
    TrainConfig.load(str(local_path), validate_paths=False)
    drive_path = CONFIG_DRIVE / name
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_path, drive_path)
    return local_path

def run_logged(args, log_path, cwd=OLMO_DIR):
    env = os.environ.copy()
    env['PYTHONPATH'] = str(OLMO_DIR) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('running:', ' '.join(str(arg) for arg in args))
    start = time.perf_counter()
    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(
            [str(arg) for arg in args], cwd=str(cwd), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        returncode = proc.wait()
    elapsed = time.perf_counter() - start
    if returncode != 0:
        tail = log_path.read_text(errors='ignore')[-4000:]
        raise RuntimeError(f'Command failed with code {returncode}. Log tail:\n{tail}')
    print('elapsed_seconds:', round(elapsed, 2), 'log:', log_path)
    return elapsed

def parse_score_log_metrics(log_path):
    text = Path(log_path).read_text(errors='ignore')
    def last_float(pattern):
        matches = re.findall(pattern, text)
        return float(matches[-1].replace(',', '')) if matches else None
    return {
        'tokens_per_second': last_float(r'throughput/device/tokens_per_second=([0-9,.]+)'),
        'batches_per_second': last_float(r'throughput/device/batches_per_second=([0-9,.]+)'),
        'peak_gpu_memory_mb': last_float(r'System/Peak GPU Memory \(MB\)=([0-9,.]+)'),
    }

def score_width(score_dir):
    files_path = Path(score_dir) / 'files.txt'
    if not files_path.exists():
        return 0
    files = [line.strip() for line in files_path.read_text().splitlines() if line.strip()]
    if not files:
        return 0
    first = Path(files[0])
    if not first.exists():
        first = Path(score_dir) / first.name
    values = first.stat().st_size // np.dtype(np.float32).itemsize
    return values // FILE_SEQS

def shard_experiment_payload(config, model_id, shard, microbatch):
    return {
        'schema_version': 3,
        'olmo_sha': OLMO_SHA, 'stage': RUN_STAGE, 'subset_id': SUBSET_ID,
        'subset_fingerprint': SUBSET_FINGERPRINT, 'config': config,
        'runtime_identity': RUNTIME_IDENTITY,
        'model_id': model_id, 'checkpoint_identity': CHECKPOINT_IDENTITIES[model_id],
        'seed': SEED, 'num_samples': NUM_SAMPLES, 'coupled_masks': True,
        'global_batch_size': GLOBAL_BATCH_SIZE, 'microbatch': int(microbatch),
        'shard': {key: int(value) for key, value in shard.items()},
    }

def shard_experiment_fingerprint(config, model_id, shard, microbatch):
    return canonical_sha256(shard_experiment_payload(config, model_id, shard, microbatch))

def score_prefix_is_finite(score_dir, expected_rows):
    files = [line.strip() for line in (Path(score_dir) / 'files.txt').read_text().splitlines() if line.strip()]
    if len(files) != 1:
        return False
    score_path = Path(files[0])
    if not score_path.exists():
        score_path = Path(score_dir) / score_path.name
    if not score_path.exists() or score_width(score_dir) != NUM_SAMPLES:
        return False
    scores = np.memmap(score_path, dtype=np.float32, mode='r', shape=(FILE_SEQS, NUM_SAMPLES))
    return bool(np.isfinite(np.asarray(scores[:expected_rows])).all())

def valid_score_output(output_dir, config, model_id, shard, microbatch):
    expected_rows = int(shard['rows'])
    output_dir = Path(output_dir)
    score_dir = output_dir / 'score'
    marker = output_dir / 'completed.json'
    index_path = score_dir / 'mmap_index.npy'
    if not marker.exists() or not index_path.exists() or not (score_dir / 'files.txt').exists():
        return False
    try:
        record = json.loads(marker.read_text())
        expected_fingerprint = shard_experiment_fingerprint(config, model_id, shard, microbatch)
        if record.get('experiment_fingerprint') != expected_fingerprint:
            return False
        if record.get('experiment') != shard_experiment_payload(config, model_id, shard, microbatch):
            return False
        index = np.memmap(index_path, dtype=np.int64, mode='r')
        if len(index) != expected_rows:
            return False
        values = np.asarray(index, dtype=np.int64)
        if len(np.unique(values)) != expected_rows:
            return False
        if values.min(initial=0) < 0 or values.max(initial=-1) >= STAGE_ROWS:
            return False
        return score_prefix_is_finite(score_dir, expected_rows)
    except Exception:
        return False

def safely_remove_invalid_shard(output_dir):
    output_dir = Path(output_dir).resolve()
    raw_root = RAW_SCORE_DRIVE.resolve()
    smoke_root = (STAGE_ROOT / '_smoke_and_benchmark').resolve()
    if raw_root not in output_dir.parents and smoke_root not in output_dir.parents:
        raise RuntimeError(f'Refusing to remove output outside isolated roots: {output_dir}')
    shutil.rmtree(output_dir)

def run_score_once(config, model_id, checkpoint_path, output_dir, shard, microbatch, console_log_interval=25):
    expected_rows = int(shard['rows'])
    output_dir = Path(output_dir)
    if valid_score_output(output_dir, config, model_id, shard, microbatch):
        print('skip valid shard:', output_dir)
        return output_dir / 'score'
    if output_dir.exists():
        print('recomputing invalid isolated shard:', output_dir)
        safely_remove_invalid_shard(output_dir)
    cfg = build_score_config(
        config, model_id, checkpoint_path, output_dir, shard, microbatch, console_log_interval
    )
    name = f"{config['config_id']}_{model_id}_{shard['start']:06d}_{shard['end']:06d}.yaml"
    cfg_path = write_config(cfg, name)
    log_path = output_dir.with_suffix('.log')
    elapsed = run_logged(
        ['torchrun', '--standalone', '--nproc_per_node=1', 'scripts/train.py', cfg_path, '--save_overwrite=true'],
        log_path,
    )
    experiment = shard_experiment_payload(config, model_id, shard, microbatch)
    marker = {
        'completed_utc': datetime.now(timezone.utc).isoformat(),
        'config_id': config['config_id'], 'model_id': model_id,
        'start': int(shard['start']), 'end': int(shard['end']), 'rows': expected_rows,
        'num_samples': NUM_SAMPLES, 'microbatch': microbatch,
        'elapsed_seconds': elapsed, 'olmo_sha': OLMO_SHA,
        'runtime_metrics': parse_score_log_metrics(log_path),
        'experiment_fingerprint': canonical_sha256(experiment), 'experiment': experiment,
    }
    (output_dir / 'completed.json').write_text(json.dumps(marker, indent=2, sort_keys=True) + '\n')
    if not valid_score_output(output_dir, config, model_id, shard, microbatch):
        raise RuntimeError(f'Score shard failed validation: {output_dir}')
    return output_dir / 'score'

def load_persisted_microbatch():
    if not RUN_STATE_PATH.exists():
        raise FileNotFoundError(f'Run state is missing; run bounded Section 7 first: {RUN_STATE_PATH}')
    state = json.loads(RUN_STATE_PATH.read_text())
    expected = {
        'schema_version': 2, 'stage': RUN_STAGE, 'olmo_sha': OLMO_SHA,
        'subset_fingerprint': SUBSET_FINGERPRINT, 'shard_rows': SHARD_ROWS,
        'num_samples': NUM_SAMPLES,
    }
    if any(state.get(key) != value for key, value in expected.items()):
        raise RuntimeError(f'Persisted run state does not match this experiment: {state}')
    microbatch = int(state['microbatch'])
    if microbatch not in (16, 32):
        raise RuntimeError(f'Unexpected persisted microbatch: {microbatch}')
    return microbatch

def shard_output_dir(config_id, model_id, shard):
    return (
        RAW_SCORE_DRIVE / config_id / model_id /
        f"shard_{shard['start']:06d}_{shard['end']:06d}"
    )

In [ ]:
# PYTHON CELL
probe_config = TARGET_CONFIGS[0]
probe_shard = {'start': 0, 'end': GLOBAL_BATCH_SIZE, 'rows': GLOBAL_BATCH_SIZE, 'data_start_step': 0}
probe_cfg = build_score_config(
    probe_config, 'prior_probe', PRIOR_CHECKPOINT, STAGE_ROOT / '_config_probe', probe_shard, MICROBATCH
)
probe_path = write_config(probe_cfg, 'config_probe.yaml')
loaded_probe = TrainConfig.load(str(probe_path), validate_paths=False)
assert loaded_probe.max_duration == 1
assert loaded_probe.data_start_step == 0
assert loaded_probe.uncertainty_scoring.num_samples == NUM_SAMPLES
assert loaded_probe.data.memmap_dtype == 'uint32'
assert loaded_probe.data.paths == [str(subset_raw)]
print('runtime config probe passed:', probe_path)

## 6. Smoke Test / GPU Gate

**Safe to rerun.** This gate is isolated to 320 rows and must pass before production scoring.

In [ ]:
# PYTHON CELL
CONTROL_CONFIG = {
    'config_id': 'dropout_trainmode_p000_smoke', 'dropout_target': 'none',
    'attention_dropout': 0.0, 'residual_dropout': 0.0, 'embedding_dropout': 0.0,
    'purpose': 'runtime and row-alignment gate',
}
smoke_shard = {'start': 0, 'end': SMOKE_ROWS, 'rows': SMOKE_ROWS, 'data_start_step': 0}
smoke_root = STAGE_ROOT / '_smoke_and_benchmark' / 'smoke'
smoke_prior = run_score_once(
    CONTROL_CONFIG, 'prior', PRIOR_CHECKPOINT, smoke_root / 'prior', smoke_shard, MICROBATCH
)
smoke_books = run_score_once(
    CONTROL_CONFIG, 'books', BOOKS_CHECKPOINT, smoke_root / 'books', smoke_shard, MICROBATCH
)
smoke_analysis = smoke_root / 'analysis'
smoke_analysis.mkdir(parents=True, exist_ok=True)
run_logged([
    sys.executable, OLMO_DIR / 'scripts/21_dropout_uncertainty_metrics.py',
    '--prior-score-dir', smoke_prior,
    '--conditional-score-dir', smoke_books,
    '--output-dir', smoke_analysis,
    '--config-id', CONTROL_CONFIG['config_id'],
    '--metadata', subset_meta,
    '--full-scores', subset_full,
    '--num-samples', NUM_SAMPLES,
    '--max-rows', SMOKE_ROWS,
    '--dropout-target', 'none',
    '--attention-dropout', 0.0,
    '--residual-dropout', 0.0,
    '--embedding-dropout', 0.0,
    '--seed', SEED,
    '--compress-npz', '--skip-parquet',
], smoke_analysis / 'aggregate.log')

smoke = np.load(smoke_analysis / f"mc_samples_{CONTROL_CONFIG['config_id']}.npz")
smoke_color = smoke['color_samples']
smoke_full = smoke['full_color_score']
smoke_spearman = float(pd.Series(smoke_color.mean(axis=1)).rank().corr(pd.Series(smoke_full).rank()))
smoke_std = float(smoke_color.std(axis=1, ddof=1).mean())
assert smoke_color.shape == (SMOKE_ROWS, NUM_SAMPLES)
assert np.isfinite(smoke_color).all()
if smoke_spearman < 0.90 or smoke_std > 1e-5:
    raise RuntimeError(
        f'Zero-dropout smoke gate failed: Spearman={smoke_spearman:.6f}, mean MC std={smoke_std:.8f}'
    )
print('smoke gate passed:', {'spearman': smoke_spearman, 'mean_mc_std': smoke_std})

## 7. Batch-Size and Shard-Size Tuning

**Benchmark only. Safe to rerun.** Each candidate is isolated and bounded to 640 rows (327,680 tokens). Valid candidates with throughput metrics are reused; candidates missing metrics are rebuilt. The fastest successful microbatch is persisted to Drive.

In [ ]:
# PYTHON CELL
benchmark_root = STAGE_ROOT / '_smoke_and_benchmark' / 'benchmark'
benchmark_shard = {'start': 0, 'end': BENCH_ROWS, 'rows': BENCH_ROWS, 'data_start_step': 0}
benchmark_results = []
for candidate in [16, 32]:
    output_dir = benchmark_root / f'prior_microbatch_{candidate}'
    try:
        marker_path = output_dir / 'completed.json'
        if marker_path.exists():
            existing_marker = json.loads(marker_path.read_text())
            existing_tps = existing_marker.get('runtime_metrics', {}).get('tokens_per_second')
            if existing_tps is None:
                print('recomputing benchmark missing throughput metrics:', output_dir)
                safely_remove_invalid_shard(output_dir)
        started = time.perf_counter()
        run_score_once(
            CONTROL_CONFIG, 'prior', PRIOR_CHECKPOINT, output_dir, benchmark_shard, candidate,
            console_log_interval=1,
        )
        wall = time.perf_counter() - started
        marker = json.loads((output_dir / 'completed.json').read_text())
        measured_tps = marker['runtime_metrics']['tokens_per_second']
        if measured_tps is None or not np.isfinite(float(measured_tps)) or float(measured_tps) <= 0:
            raise RuntimeError(f'Benchmark did not record valid device throughput: {marker["runtime_metrics"]}')
        result = {
            'microbatch': candidate, 'status': 'ok', 'cell_wall_seconds': wall,
            'measured_compute_seconds': float(marker['elapsed_seconds']),
            'tokens_per_second': float(measured_tps),
            'rows': BENCH_ROWS, 'tokens': BENCH_ROWS * SEQ_LEN,
        }
        benchmark_results.append(result)
    except Exception as exc:
        benchmark_results.append({'microbatch': candidate, 'status': f'failed: {exc}'})
        print('stopping after first failed candidate')
        break
successful_results = [result for result in benchmark_results if result['status'] == 'ok']
if not successful_results:
    raise RuntimeError(f'No microbatch candidate succeeded: {benchmark_results}')
selected_result = max(successful_results, key=lambda result: result['tokens_per_second'])
selected_microbatch = int(selected_result['microbatch'])
MICROBATCH = selected_microbatch
measured_tps = float(selected_result['tokens_per_second'])
if not np.isfinite(measured_tps) or measured_tps <= 0:
    raise RuntimeError(f'Benchmark did not record valid device throughput: {selected_result}')
startup_seconds = max(
    0.0, selected_result['measured_compute_seconds'] - selected_result['tokens'] / measured_tps
)
production_jobs = len(TARGET_CONFIGS) * 2 * len(SHARDS)
production_tokens = len(TARGET_CONFIGS) * 2 * STAGE_ROWS * SEQ_LEN
estimated_minutes = (production_tokens / measured_tps + production_jobs * startup_seconds) / 60
run_state = {
    'schema_version': 2, 'created_utc': datetime.now(timezone.utc).isoformat(),
    'stage': RUN_STAGE, 'olmo_sha': OLMO_SHA, 'subset_fingerprint': SUBSET_FINGERPRINT,
    'num_samples': NUM_SAMPLES, 'microbatch': MICROBATCH, 'shard_rows': SHARD_ROWS,
    'benchmark': selected_result,
}
RUN_STATE_PATH.write_text(json.dumps(run_state, indent=2, sort_keys=True) + '\n')
print('benchmark results:', benchmark_results)
print('selected microbatch:', MICROBATCH)
print('ETA formula: total production tokens / measured tokens/sec + jobs * measured startup overhead')
print('measured tokens/sec:', round(measured_tps, 1), 'startup seconds/job:', round(startup_seconds, 1))
print('conservative full-stage estimate minutes:', round(estimated_minutes, 1))
print('production shards:', len(SHARDS), 'rows per full shard:', SHARD_ROWS)

## 8. Full Resumable Stage Run

**Full run. Safe to rerun.** Valid fingerprint-matched shards are skipped; invalid isolated shards are rebuilt.

In [ ]:
# PYTHON CELL
MICROBATCH = load_persisted_microbatch()
print('using persisted production microbatch:', MICROBATCH)

for config in TARGET_CONFIGS:
    for model_id, checkpoint in [('prior', PRIOR_CHECKPOINT), ('books', BOOKS_CHECKPOINT)]:
        for shard in SHARDS:
            run_score_once(
                config,
                model_id,
                checkpoint,
                shard_output_dir(config['config_id'], model_id, shard),
                shard,
                MICROBATCH,
            )
print('all expected production shards completed')

## 9. Resume After Disconnect / Status

**Safe to rerun. Fresh-runtime checklist:** rerun Sections 1-5 in order; do not rerun Sections 6-7 when `run_state.json` already exists; run this status cell; rerun Section 8 to repair only invalid shards; then continue with Sections 10-11.

In [ ]:
# PYTHON CELL
MICROBATCH = load_persisted_microbatch()
status_rows = []
for config in TARGET_CONFIGS:
    for model_id in ['prior', 'books']:
        for shard in SHARDS:
            output_dir = shard_output_dir(config['config_id'], model_id, shard)
            status_rows.append({
                'config_id': config['config_id'],
                'model_id': model_id,
                'start': shard['start'],
                'end': shard['end'],
                'rows': shard['rows'],
                'valid': valid_score_output(output_dir, config, model_id, shard, MICROBATCH),
                'output_dir': str(output_dir),
            })
status = pd.DataFrame(status_rows)
print('valid shards:', int(status['valid'].sum()), '/', len(status))
print(status.groupby(['config_id', 'model_id'])['valid'].agg(['sum', 'count']).to_string())
incomplete = status.loc[~status['valid'], ['config_id', 'model_id', 'start', 'end']]
if len(incomplete):
    print('\nIncomplete shards. Rerun Section 8; valid shards will be skipped.')
    print(incomplete.to_string(index=False))
else:
    print('\nAll shards are complete. Continue to Section 10.')
status

## 10. Metrics, Strategy Sweep, and Persisted Outputs

**Safe to rerun.** This cell validates the complete raw grid, removes only an invalid per-config analysis directory, regenerates it, and validates artifact contents before continuing.

In [ ]:
# PYTHON CELL
EXPECTED_SELECTED_FILES = 64
EXPECTED_PAIRWISE_METRIC_ROWS = 96
EXPECTED_FULL_POOL_METRIC_ROWS = 64
EXPECTED_OVERLAP_ROWS = 480

def score_dirs_for(config_id, model_id):
    return [shard_output_dir(config_id, model_id, shard) / 'score' for shard in SHARDS]

def analysis_context_payload(config):
    return {
        'schema_version': 1, 'olmo_sha': OLMO_SHA, 'stage': RUN_STAGE,
        'subset_fingerprint': SUBSET_FINGERPRINT, 'config': config,
        'seed': SEED, 'num_samples': NUM_SAMPLES, 'microbatch': MICROBATCH,
        'raw_shard_fingerprints': {
            model_id: [shard_experiment_fingerprint(config, model_id, shard, MICROBATCH) for shard in SHARDS]
            for model_id in ('prior', 'books')
        },
    }

def validate_raw_grid(config):
    indexes = {}
    for model_id in ('prior', 'books'):
        model_indexes = []
        for shard in SHARDS:
            output_dir = shard_output_dir(config['config_id'], model_id, shard)
            if not valid_score_output(output_dir, config, model_id, shard, MICROBATCH):
                raise RuntimeError(f'Invalid raw shard: {output_dir}')
            index = np.memmap(output_dir / 'score' / 'mmap_index.npy', dtype=np.int64, mode='r')
            model_indexes.append(np.asarray(index, dtype=np.int64))
        combined = np.concatenate(model_indexes)
        if not np.array_equal(np.sort(combined), np.arange(STAGE_ROWS, dtype=np.int64)):
            raise RuntimeError(f"{config['config_id']} {model_id}: shard indexes do not cover the subset exactly once")
        indexes[model_id] = combined
    if not np.array_equal(indexes['prior'], indexes['books']):
        raise RuntimeError(f"{config['config_id']}: prior and Books row order differs")

def validate_analysis(config):
    config_id = config['config_id']
    analysis = STAGE_ROOT / config_id / 'analysis'
    strategy = analysis / 'strategy'
    paths = {
        'npz': analysis / f'mc_samples_{config_id}.npz',
        'parquet': analysis / f'mc_samples_{config_id}.parquet',
        'manifest': analysis / f'mc_samples_{config_id}_manifest.json',
        'context': analysis / 'analysis_context.json',
        'summary': analysis / 'color_distribution_summary.parquet',
        'metrics': strategy / 'strategy_sweep_metrics.csv',
        'overlap': strategy / 'strategy_selection_overlap.csv',
    }
    missing = [str(path) for path in paths.values() if not path.is_file() or path.stat().st_size <= 0]
    if missing:
        raise FileNotFoundError('Missing or empty analysis files: ' + ', '.join(missing))
    expected_context = analysis_context_payload(config)
    context = json.loads(paths['context'].read_text())
    if context.get('fingerprint') != canonical_sha256(expected_context) or context.get('experiment') != expected_context:
        raise ValueError(f'{config_id}: stale analysis context')
    manifest = json.loads(paths['manifest'].read_text())
    expected_manifest = {
        'config_id': config_id, 'rows': STAGE_ROWS, 'num_samples': NUM_SAMPLES,
        'dropout_target': config['dropout_target'], 'seed': SEED, 'coupled_masks': True,
        'attention_dropout': config['attention_dropout'],
        'residual_dropout': config['residual_dropout'],
        'embedding_dropout': config['embedding_dropout'],
    }
    for key, expected in expected_manifest.items():
        if manifest.get(key) != expected:
            raise ValueError(f'{config_id}: manifest {key}={manifest.get(key)!r}, expected {expected!r}')
    required_arrays = {
        'seq_idx', 'score_index', 'pool_name', 'prior_losses', 'conditional_losses',
        'color_samples', 'utility_samples', 'metadata_json', 'full_color_score',
    }
    with np.load(paths['npz'], allow_pickle=False) as raw:
        if not required_arrays.issubset(raw.files):
            raise ValueError(f'{config_id}: NPZ missing {sorted(required_arrays.difference(raw.files))}')
        score_index = raw['score_index'].astype(np.int64)
        if not np.array_equal(np.sort(score_index), np.arange(STAGE_ROWS, dtype=np.int64)):
            raise ValueError(f'{config_id}: score_index is not a complete permutation')
        prior = raw['prior_losses']
        conditional = raw['conditional_losses']
        color = raw['color_samples']
        if prior.shape != (STAGE_ROWS, NUM_SAMPLES) or conditional.shape != prior.shape or color.shape != prior.shape:
            raise ValueError(f'{config_id}: invalid sample tensor shapes')
        if not np.isfinite(prior).all() or not np.isfinite(conditional).all() or not np.isfinite(color).all():
            raise ValueError(f'{config_id}: non-finite MC samples')
        if not np.allclose(color, conditional - prior, rtol=1e-5, atol=1e-6):
            raise ValueError(f'{config_id}: color samples do not equal conditional-prior')
        if len(raw['seq_idx']) != STAGE_ROWS or len(np.unique(raw['seq_idx'])) != STAGE_ROWS:
            raise ValueError(f'{config_id}: seq_idx is incomplete or duplicated')
        seq_idx = raw['seq_idx'].astype(np.int64)
        if not np.isfinite(raw['full_color_score']).all():
            raise ValueError(f'{config_id}: non-finite full-color reference')
    summary = pd.read_parquet(paths['summary'])
    summary_columns = {'seq_idx', 'score_index', 'pool_name', 'mean_color', 'std_color', 'full_color_score'}
    if len(summary) != STAGE_ROWS or not summary_columns.issubset(summary.columns):
        raise ValueError(f'{config_id}: invalid summary schema or row count')
    if not np.array_equal(np.sort(summary['score_index'].to_numpy(dtype=np.int64)), np.arange(STAGE_ROWS)):
        raise ValueError(f'{config_id}: summary score_index is incomplete')
    metrics = pd.read_csv(paths['metrics'])
    pairwise = metrics[metrics['metric_scope'] == 'pairwise']
    full_pool = metrics[metrics['metric_scope'] == 'full_pool']
    if len(pairwise) != EXPECTED_PAIRWISE_METRIC_ROWS or len(full_pool) != EXPECTED_FULL_POOL_METRIC_ROWS:
        raise ValueError(f'{config_id}: unexpected metric rows pairwise={len(pairwise)} full={len(full_pool)}')
    expected_tasks = {'hp_vs_hn', 'hp_vs_rn', 'hp_vs_tn', 'rp_vs_hn', 'rp_vs_rn', 'rp_vs_tn'}
    if set(pairwise['task_id']) != expected_tasks or metrics['strategy'].nunique() != 16:
        raise ValueError(f'{config_id}: incomplete task or strategy grid')
    if not np.isfinite(pairwise['roc_auc']).all() or not np.isfinite(full_pool['recall_vs_full']).all():
        raise ValueError(f'{config_id}: non-finite strategy metrics')
    hp_mean = pairwise[(pairwise['strategy'] == 'mean') & (pairwise['task_id'] == 'hp_vs_hn')]
    rate_mean = full_pool[(full_pool['strategy'] == 'mean') & np.isclose(full_pool['selection_rate'], 1 / 64)]
    if len(hp_mean) != 1 or len(rate_mean) != 1:
        raise ValueError(f'{config_id}: required mean-strategy metric rows are missing or duplicated')
    overlap = pd.read_csv(paths['overlap'])
    if len(overlap) != EXPECTED_OVERLAP_ROWS or not np.isfinite(overlap['jaccard']).all():
        raise ValueError(f'{config_id}: invalid strategy overlap table')
    selected = sorted((strategy / 'strategy_selected_indices').glob('*.npy'))
    expected_sizes = set(full_pool['n_selected'].astype(int).tolist())
    if len(selected) != EXPECTED_SELECTED_FILES:
        raise ValueError(f'{config_id}: expected {EXPECTED_SELECTED_FILES} selected arrays, found {len(selected)}')
    for path in selected:
        values = np.load(path, allow_pickle=False)
        if values.ndim != 1 or not np.issubdtype(values.dtype, np.integer):
            raise ValueError(f'{config_id}: invalid selected array {path.name}')
        if len(values) not in expected_sizes or len(np.unique(values)) != len(values):
            raise ValueError(f'{config_id}: invalid selected IDs in {path.name}')
        if not np.isin(values, seq_idx).all():
            raise ValueError(f'{config_id}: selected IDs fall outside the fixed subset in {path.name}')
    return True

def safely_remove_invalid_analysis(analysis):
    analysis = Path(analysis).resolve()
    expected_parent = STAGE_ROOT.resolve() / analysis.parent.name
    if analysis.name != 'analysis' or analysis.parent != expected_parent:
        raise RuntimeError(f'Refusing to remove non-isolated analysis path: {analysis}')
    print('removing invalid isolated analysis:', analysis)
    shutil.rmtree(analysis)

MICROBATCH = load_persisted_microbatch()
for config in TARGET_CONFIGS:
    validate_raw_grid(config)

for config in TARGET_CONFIGS:
    config_id = config['config_id']
    analysis = STAGE_ROOT / config_id / 'analysis'
    strategy = analysis / 'strategy'
    try:
        validate_analysis(config)
        print('skip validated analysis:', config_id)
        continue
    except Exception as exc:
        print('analysis requires repair:', config_id, type(exc).__name__, exc)
    if analysis.exists():
        safely_remove_invalid_analysis(analysis)
    analysis.mkdir(parents=True, exist_ok=True)
    aggregate_args = [
        sys.executable, OLMO_DIR / 'scripts/21_dropout_uncertainty_metrics.py',
        '--prior-score-dir', *score_dirs_for(config_id, 'prior'),
        '--conditional-score-dir', *score_dirs_for(config_id, 'books'),
        '--output-dir', analysis, '--config-id', config_id,
        '--metadata', subset_meta, '--full-scores', subset_full,
        '--num-samples', NUM_SAMPLES, '--max-rows', STAGE_ROWS,
        '--dropout-target', config['dropout_target'],
        '--attention-dropout', config['attention_dropout'],
        '--residual-dropout', config['residual_dropout'],
        '--embedding-dropout', config['embedding_dropout'],
        '--seed', SEED, '--compress-npz',
    ]
    run_logged(aggregate_args, analysis / 'aggregate.log')
    run_logged([
        sys.executable, OLMO_DIR / 'scripts/22_dropout_strategy_sweep.py',
        '--mc-samples', analysis / f'mc_samples_{config_id}.npz',
        '--summary', analysis / 'color_distribution_summary.parquet',
        '--output-dir', strategy, '--tau64-cutoff', TAU64_CUTOFF,
    ], analysis / 'strategy.log')
    experiment = analysis_context_payload(config)
    (analysis / 'analysis_context.json').write_text(json.dumps({
        'fingerprint': canonical_sha256(experiment), 'experiment': experiment,
    }, indent=2, sort_keys=True) + '\n')
    validate_analysis(config)
print('all analyses passed content validation')

## 11. Output Review, Promotion Gate, and Download Bundle

In [ ]:
# PYTHON CELL
import matplotlib.pyplot as plt

PAIRWISE_TASKS = {
    'hp_vs_hn': ('hard_positive', 'hard_negative'),
    'hp_vs_rn': ('hard_positive', 'random_negative'),
    'hp_vs_tn': ('hard_positive', 'tail_negative'),
    'rp_vs_hn': ('random_positive', 'hard_negative'),
    'rp_vs_rn': ('random_positive', 'random_negative'),
    'rp_vs_tn': ('random_positive', 'tail_negative'),
}

def selected_mask(scores, k):
    mask = np.zeros(len(scores), dtype=bool)
    k = min(max(1, int(k)), len(scores))
    mask[np.argpartition(scores, k - 1)[:k]] = True
    return mask

def uncertainty_error_ratio(mean_color, std_color, pool_name):
    low_rates, high_rates = [], []
    for positive_pool, negative_pool in PAIRWISE_TASKS.values():
        task_mask = (pool_name == positive_pool) | (pool_name == negative_pool)
        labels = pool_name[task_mask] == positive_pool
        scores = mean_color[task_mask]
        uncertainty = std_color[task_mask]
        if not len(scores) or not labels.any():
            continue
        errors = selected_mask(scores, int(labels.sum())) != labels
        deciles = pd.qcut(uncertainty, 10, labels=False, duplicates='drop')
        if pd.isna(deciles).all():
            continue
        deciles = np.asarray(deciles, dtype=np.int64)
        low_rates.append(float(errors[deciles == deciles.min()].mean()))
        high_rates.append(float(errors[deciles == deciles.max()].mean()))
    low = float(np.mean(low_rates)) if low_rates else float('nan')
    high = float(np.mean(high_rates)) if high_rates else float('nan')
    ratio = high / low if low and np.isfinite(low) else float('nan')
    return low, high, ratio

def runtime_records(config_id):
    records = []
    for model_id in ['prior', 'books']:
        for shard in SHARDS:
            marker_path = shard_output_dir(config_id, model_id, shard) / 'completed.json'
            marker = json.loads(marker_path.read_text())
            runtime = marker.get('runtime_metrics', {})
            records.append({
                'config_id': config_id, 'model_id': model_id,
                'start': shard['start'], 'end': shard['end'], 'rows': shard['rows'],
                'elapsed_seconds': float(marker['elapsed_seconds']),
                'tokens_per_second': runtime.get('tokens_per_second'),
                'batches_per_second': runtime.get('batches_per_second'),
                'peak_gpu_memory_mb': runtime.get('peak_gpu_memory_mb'),
                'microbatch': int(marker['microbatch']),
            })
    return records

summary_rows = []
all_runtime_rows = []
for config in TARGET_CONFIGS:
    cid = config['config_id']
    validate_analysis(config)
    analysis = STAGE_ROOT / cid / 'analysis'
    raw = np.load(analysis / f'mc_samples_{cid}.npz')
    color = raw['color_samples']
    score_index = raw['score_index'].astype(np.int64)
    full_color = raw['full_color_score']
    pool_name = raw['pool_name'].astype(str)
    if color.shape != (STAGE_ROWS, NUM_SAMPLES) or not np.isfinite(color).all():
        raise RuntimeError(f'{cid}: invalid raw MC sample tensor {color.shape}')
    if len(np.unique(score_index)) != STAGE_ROWS or not np.array_equal(np.sort(score_index), np.arange(STAGE_ROWS)):
        raise RuntimeError(f'{cid}: score_index does not cover the fixed subset exactly once')

    mean_color = color.mean(axis=1)
    std_color = color.std(axis=1, ddof=1)
    spearman = float(pd.Series(mean_color).rank().corr(pd.Series(full_color).rank()))
    pearson = float(np.corrcoef(mean_color, full_color)[0, 1])
    metrics = pd.read_csv(analysis / 'strategy' / 'strategy_sweep_metrics.csv')
    pair_mean = metrics[(metrics['metric_scope'] == 'pairwise') & (metrics['strategy'] == 'mean')]
    full_mean = metrics[(metrics['metric_scope'] == 'full_pool') & (metrics['strategy'] == 'mean')]
    mean_auc = float(pair_mean['roc_auc'].mean())
    hp_auc = float(pair_mean.loc[pair_mean['task_id'] == 'hp_vs_hn', 'roc_auc'].iloc[0])
    recall = float(full_mean.loc[np.isclose(full_mean['selection_rate'], 1 / 64), 'recall_vs_full'].iloc[0])
    low_error, high_error, error_ratio = uncertainty_error_ratio(mean_color, std_color, pool_name)
    q05 = np.quantile(color, 0.05, axis=1)
    q95 = np.quantile(color, 0.95, axis=1)
    uncertain_fraction = float(((q05 <= TAU64_CUTOFF) & (q95 > TAU64_CUTOFF)).mean())

    if spearman <= 0.20:
        decision, reason = 'stop', 'Spearman is at or below the 0.20 hard gate'
    elif recall <= 2 / 64:
        decision, reason = 'stop', '1/64 recall is within 2x random overlap'
    elif mean_auc <= 0.55 and hp_auc <= 0.55:
        decision, reason = 'stop', 'pairwise AUC remains near chance'
    elif spearman >= 0.50 and (mean_auc >= 0.75 or hp_auc >= 0.60 or recall >= 0.10 or error_ratio >= 1.25):
        decision, reason = 'promote', 'rank is preserved and at least one promotion target is met'
    else:
        decision, reason = 'review', 'hard gates pass but promotion targets are not clearly met'

    runtimes = pd.DataFrame(runtime_records(cid))
    all_runtime_rows.extend(runtimes.to_dict('records'))
    runtime_columns = ['elapsed_seconds', 'tokens_per_second', 'batches_per_second', 'peak_gpu_memory_mb']
    if runtimes[runtime_columns].isna().any().any() or not np.isfinite(runtimes[runtime_columns]).all().all():
        raise RuntimeError(f'{cid}: incomplete runtime metrics in completion markers')
    if runtimes['microbatch'].nunique() != 1 or int(runtimes['microbatch'].iloc[0]) != MICROBATCH:
        raise RuntimeError(f'{cid}: inconsistent runtime microbatch records')
    summary_rows.append({
        'config_id': cid, 'dropout_target': config['dropout_target'],
        'attention_dropout': config['attention_dropout'],
        'residual_dropout': config['residual_dropout'],
        'embedding_dropout': config['embedding_dropout'],
        'rows': STAGE_ROWS, 'K': NUM_SAMPLES,
        'spearman_mean_vs_full': spearman, 'pearson_mean_vs_full': pearson,
        'mean_pairwise_auc': mean_auc, 'hp_vs_hn_auc': hp_auc,
        'recall_vs_full_1_64': recall, 'mean_mc_std': float(std_color.mean()),
        'low_uncertainty_error_rate': low_error, 'high_uncertainty_error_rate': high_error,
        'error_rate_ratio_high_vs_low': error_ratio,
        'triage_uncertain_fraction': uncertain_fraction,
        'runtime_seconds': float(runtimes['elapsed_seconds'].sum()),
        'tokens_per_second_mean': float(runtimes['tokens_per_second'].dropna().mean()),
        'batches_per_second_mean': float(runtimes['batches_per_second'].dropna().mean()),
        'peak_gpu_memory_mb': float(runtimes['peak_gpu_memory_mb'].dropna().max()),
        'global_batch_size': GLOBAL_BATCH_SIZE, 'shard_rows': SHARD_ROWS,
        'microbatch': int(runtimes['microbatch'].iloc[0]),
        'shard_count': int(len(runtimes)),
        'prior_shard_count': int((runtimes['model_id'] == 'prior').sum()),
        'conditional_shard_count': int((runtimes['model_id'] == 'books').sum()),
        'promotion_decision': decision, 'decision_reason': reason,
    })

stage_summary = pd.DataFrame(summary_rows)
summary_path = STAGE_ROOT / 'stage_summary.csv'
stage_summary.to_csv(summary_path, index=False)
runtime_table = pd.DataFrame(all_runtime_rows)
runtime_path = STAGE_ROOT / 'runtime_summary.csv'
runtime_table.to_csv(runtime_path, index=False)
promoted = stage_summary.loc[stage_summary['promotion_decision'] == 'promote'].copy()
promoted = promoted.sort_values(
    ['spearman_mean_vs_full', 'mean_pairwise_auc', 'recall_vs_full_1_64'], ascending=False
)
promoted_ids = promoted['config_id'].tolist()
recommended_stage_c = promoted_ids[:2] if RUN_STAGE == 'stage_b_100k' else []
acceptance = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'stage_id': RUN_STAGE, 'stage_rows': STAGE_ROWS, 'num_samples': NUM_SAMPLES,
    'olmo_sha': OLMO_SHA, 'subset_manifest': str(SUBSET_MANIFEST),
    'stage_passed': bool(promoted_ids),
    'promoted_config_ids': promoted_ids,
    'recommended_stage_c_config_ids': recommended_stage_c,
    'hard_gate_thresholds': {
        'spearman_gt': 0.20, 'recall_1_64_gt': 2 / 64, 'near_chance_auc_max': 0.55,
    },
    'config_decisions': stage_summary[
        ['config_id', 'promotion_decision', 'decision_reason']
    ].to_dict('records'),
    'final_interpretation_status': (
        'ready_for_local_cross_baseline_report' if RUN_STAGE == 'stage_c_500k'
        else 'Stage C allowed only for recommended or explicitly promoted IDs'
    ),
}
acceptance_path = STAGE_ROOT / 'stage_acceptance.json'
acceptance_path.write_text(json.dumps(acceptance, indent=2, sort_keys=True) + '\n')

comparison_contract = {
    'deterministic_full_scores': str(FULL_SCORES_DRIVE),
    'previous_broad_dropout_config': 'dropout_k8_p005',
    'previous_broad_dropout_analysis': str(BROAD_DROPOUT_ANALYSIS),
    'previous_broad_dropout_alignment_note': (
        'Reindex its compact metadata/full-score references by score_index before final comparison.'
    ),
    'pair_mid2_cascade_scores': str(PAIR_MID2_DRIVE),
    'targeted_stage_root': str(STAGE_ROOT),
    'local_extract_root': f'results/dropout-uncertainty-targeted/{SUBSET_ID}',
    'local_report_command': (
        'python scripts/22_targeted_dropout_cross_stage_report.py '
        '--stage-b-root results/dropout-uncertainty-targeted/stage_b_100k '
        + ('--stage-c-root results/dropout-uncertainty-targeted/stage_c_500k ' if RUN_STAGE == 'stage_c_500k' else '')
        + '--output-dir reports/dropout-uncertainty-targeted-ladder-final'
    ),
    'final_local_report_command': (
        'python scripts/22_targeted_dropout_cross_stage_report.py '
        '--stage-b-root results/dropout-uncertainty-targeted/stage_b_100k '
        '--stage-c-root results/dropout-uncertainty-targeted/stage_c_500k '
        '--output-dir reports/dropout-uncertainty-targeted-ladder-final'
    ),
    'required_final_conclusions': [
        'standalone selection', 'cascade candidate generation', 'triage routing',
        'uncertainty diagnostics', 'none of the above',
    ],
}
comparison_path = STAGE_ROOT / 'final_comparison_contract.json'
comparison_path.write_text(json.dumps(comparison_contract, indent=2, sort_keys=True) + '\n')

REPORT_DRIVE.mkdir(parents=True, exist_ok=True)
plot_frame = stage_summary.set_index('config_id')
fig, ax = plt.subplots(figsize=(max(8, len(plot_frame) * 1.35), 5))
plot_frame[['spearman_mean_vs_full', 'mean_pairwise_auc', 'hp_vs_hn_auc', 'recall_vs_full_1_64']].plot(
    kind='bar', ax=ax
)
ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Metric value')
ax.set_xlabel('')
ax.set_title(f'{RUN_STAGE}: targeted-dropout quality metrics')
ax.legend(loc='best', fontsize=8)
fig.tight_layout()
quality_figure = REPORT_DRIVE / 'quality_metrics.png'
fig.savefig(quality_figure, dpi=160)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_frame['runtime_seconds'].div(3600).plot(kind='bar', ax=axes[0], color='#3b6ea8')
plot_frame['tokens_per_second_mean'].plot(kind='bar', ax=axes[1], color='#c05a47')
axes[0].set_title('Total scoring runtime')
axes[0].set_ylabel('GPU hours')
axes[1].set_title('Mean device throughput')
axes[1].set_ylabel('Tokens / second')
for ax in axes:
    ax.set_xlabel('')
fig.tight_layout()
runtime_figure = REPORT_DRIVE / 'runtime_metrics.png'
fig.savefig(runtime_figure, dpi=160)
plt.close(fig)

decision_lines = [
    f"- `{row.config_id}`: **{row.promotion_decision}** - {row.decision_reason}"
    for row in stage_summary.itertuples(index=False)
]
report_md = REPORT_DRIVE / 'stage_report.md'
report_md.write_text(
    f"# Targeted Dropout Ladder: {RUN_STAGE}\n\n"
    f"- Rows: {STAGE_ROWS:,}\n- MC samples: {NUM_SAMPLES}\n- OLMo commit: `{OLMO_SHA}`\n"
    f"- Stage passed: {acceptance['stage_passed']}\n\n## Decisions\n\n"
    + '\n'.join(decision_lines)
    + '\n\n## Summary\n\n```csv\n'
    + stage_summary.to_csv(index=False, float_format='%.6g')
    + '```\n\n## Figures\n\n![Quality metrics](quality_metrics.png)\n\n'
    + '![Runtime metrics](runtime_metrics.png)\n\n'
    + '## Local Handoff\n\n```bash\n'
    + comparison_contract['local_report_command']
    + '\n```\n',
    encoding='utf-8',
)
report_html = REPORT_DRIVE / 'stage_report.html'
report_html.write_text(
    '<!doctype html><meta charset="utf-8"><title>Targeted dropout stage report</title>'
    '<style>body{font:15px system-ui;max-width:1200px;margin:32px auto;padding:0 20px}'
    'table{border-collapse:collapse;width:100%;font-size:12px}th,td{border:1px solid #ccc;padding:5px}'
    'img{max-width:100%;height:auto}code{background:#eee;padding:2px 4px}</style>'
    f'<h1>Targeted Dropout Ladder: {RUN_STAGE}</h1>'
    f'<p>Rows: {STAGE_ROWS:,}; MC samples: {NUM_SAMPLES}; stage passed: {acceptance["stage_passed"]}</p>'
    + stage_summary.to_html(index=False, float_format=lambda value: f'{value:.6g}')
    + '<h2>Quality</h2><img src="quality_metrics.png" alt="Quality metrics">'
    + '<h2>Runtime</h2><img src="runtime_metrics.png" alt="Runtime metrics">'
    + f'<h2>Local handoff</h2><pre>{comparison_contract["local_report_command"]}</pre>',
    encoding='utf-8',
)
report_manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(), 'stage_id': RUN_STAGE,
    'artifacts': [path.name for path in [report_md, report_html, quality_figure, runtime_figure]],
}
(REPORT_DRIVE / 'report_manifest.json').write_text(
    json.dumps(report_manifest, indent=2, sort_keys=True) + '\n'
)

print(stage_summary.to_string(index=False))
print('\nacceptance:', json.dumps(acceptance, indent=2, sort_keys=True))
if RUN_STAGE == 'stage_b_100k' and not acceptance['stage_passed']:
    print('\nSTOP: Stage B did not promote any target. Do not enable Stage C.')
elif RUN_STAGE == 'stage_b_100k':
    print('\nStage C recommendation:', recommended_stage_c)

In [ ]:
# PYTHON CELL
import zipfile
from google.colab import files

AUTO_DOWNLOAD = False
archive_path = Path('/content') / f'targeted_dropout_ladder_{SUBSET_ID}.zip'
bundle_manifest_path = Path('/content') / f'targeted_dropout_ladder_{SUBSET_ID}_bundle_manifest.json'
required_bundle_files = []

def require_bundle_file(path, arcname):
    path = Path(path)
    if not path.is_file() or path.stat().st_size <= 0:
        raise FileNotFoundError(f'Required bundle artifact is missing or empty: {path}')
    required_bundle_files.append((path, arcname))

for path, arcname in [
    (SUBSET_MANIFEST, 'subset_manifest.json'),
    (SOURCE_ROWS_DRIVE, 'subset_source_rows.npy'),
    (RUN_STATE_PATH, 'run_state.json'),
    (STAGE_ROOT / 'shard_plan.json', 'shard_plan.json'),
    (STAGE_ROOT / 'stage_summary.csv', 'stage_summary.csv'),
    (STAGE_ROOT / 'runtime_summary.csv', 'runtime_summary.csv'),
    (STAGE_ROOT / 'stage_acceptance.json', 'stage_acceptance.json'),
    (STAGE_ROOT / 'final_comparison_contract.json', 'final_comparison_contract.json'),
]:
    require_bundle_file(path, arcname)
for name in ['stage_report.md', 'stage_report.html', 'quality_metrics.png', 'runtime_metrics.png', 'report_manifest.json']:
    require_bundle_file(REPORT_DRIVE / name, f'report/{name}')

for config in TARGET_CONFIGS:
    validate_analysis(config)
    cid = config['config_id']
    analysis = STAGE_ROOT / cid / 'analysis'
    strategy = analysis / 'strategy'
    base = f'analysis/{cid}'
    for name in [
        'aggregate.log', 'strategy.log', 'analysis_context.json',
        'color_distribution_summary.parquet', f'mc_samples_{cid}.npz',
        f'mc_samples_{cid}.parquet', f'mc_samples_{cid}_manifest.json',
    ]:
        require_bundle_file(analysis / name, f'{base}/{name}')
    for name in ['strategy_sweep_metrics.csv', 'strategy_selection_overlap.csv']:
        require_bundle_file(strategy / name, f'{base}/strategy/{name}')
    selected_files = sorted((strategy / 'strategy_selected_indices').glob('*.npy'))
    if len(selected_files) != EXPECTED_SELECTED_FILES:
        raise RuntimeError(f'{cid}: expected {EXPECTED_SELECTED_FILES} selected arrays before bundling')
    for selected in selected_files:
        require_bundle_file(
            selected, f'{base}/strategy/strategy_selected_indices/{selected.name}'
        )
    for model_id in ['prior', 'books']:
        for shard in SHARDS:
            output_dir = shard_output_dir(cid, model_id, shard)
            shard_name = f"shard_{shard['start']:06d}_{shard['end']:06d}"
            raw_base = f'raw_score_provenance/{cid}/{model_id}/{shard_name}'
            require_bundle_file(output_dir / 'completed.json', f'{raw_base}/completed.json')
            require_bundle_file(output_dir.with_suffix('.log'), f'{raw_base}.log')
            require_bundle_file(output_dir / 'config.yaml', f'{raw_base}/config.yaml')
            runtime_name = f"{cid}_{model_id}_{shard['start']:06d}_{shard['end']:06d}.yaml"
            require_bundle_file(CONFIG_DRIVE / runtime_name, f'runtime_configs/{runtime_name}')

archive_names = [arcname for _, arcname in required_bundle_files]
if len(archive_names) != len(set(archive_names)):
    raise RuntimeError('Bundle contains duplicate archive names')
bundle_manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'stage_id': RUN_STAGE, 'stage_rows': STAGE_ROWS, 'num_samples': NUM_SAMPLES,
    'olmo_sha': OLMO_SHA, 'target_configs': TARGET_CONFIGS,
    'stage_root': str(STAGE_ROOT),
    'files': [
        {'source': str(path), 'archive': arcname, 'bytes': path.stat().st_size}
        for path, arcname in required_bundle_files
    ],
    'excludes': ['model checkpoints', 'token arrays', 'raw score memmaps'],
    'local_extract_root': f'results/dropout-uncertainty-targeted/{SUBSET_ID}',
    'local_report_command': comparison_contract['local_report_command'],
    'final_local_report_command': comparison_contract['final_local_report_command'],
}
bundle_manifest_path.write_text(json.dumps(bundle_manifest, indent=2, sort_keys=True) + '\n')

with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zf.write(bundle_manifest_path, 'bundle_manifest.json')
    for path, arcname in required_bundle_files:
        zf.write(path, arcname)

expected_names = {'bundle_manifest.json', *archive_names}
with zipfile.ZipFile(archive_path, 'r') as zf:
    actual_names = set(zf.namelist())
    if actual_names != expected_names:
        raise RuntimeError(f'Bundle member mismatch: missing={expected_names - actual_names}, extra={actual_names - expected_names}')
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f'Bundle CRC check failed for {bad_member}')

print('verified bundle files:', len(expected_names))
print('wrote bundle:', archive_path, 'size MB:', round(archive_path.stat().st_size / 1e6, 2))
print('extract locally to:', bundle_manifest['local_extract_root'])
print('then run:', bundle_manifest['local_report_command'])
print('Set AUTO_DOWNLOAD=True and rerun this cell to download automatically.')
if AUTO_DOWNLOAD:
    files.download(str(archive_path))